# RGB Benchmark Results → LaTeX Tables

In [ ]:
import pandas as pd
import numpy as np

UPERNET_LONG = "../rgb_upernet_results_long.csv"
MLP_LONG     = "../rgb_mlp_results_long.csv"

# Datasets in display order
DATASETS = [
    "hlsburnscars", "sen1floods11", "mados", "pastis",
    "dynamicen", "croptypemapping", "spacenet7cd",
    "ai4smallfarms", "fivebillionpixels",
]

DATASET_LABELS = {
    "hlsburnscars":      "HLS-BurnScars",
    "sen1floods11":      "Sen1Floods11",
    "mados":             "MADOS",
    "pastis":            "PASTIS",
    "dynamicen":         "DynamicEarthNet",
    "croptypemapping":   "CropTypeMapping",
    "spacenet7cd":       "SpaceNet7-CD",
    "ai4smallfarms":     "AI4SmallFarms",
    "fivebillionpixels": "FiveBillionPixels",
}

# Which decoder to use per dataset (UPerNet experiments)
UPERNET_DECODER = {
    "hlsburnscars":      "seg_upernet",
    "sen1floods11":      "seg_upernet",
    "mados":             "seg_upernet",
    "pastis":            "seg_upernet_mt_ltae",
    "dynamicen":         "seg_upernet_mt_ltae",
    "croptypemapping":   "seg_upernet_mt_ltae",
    "spacenet7cd":       "seg_siamupernet_conc",
    "ai4smallfarms":     "seg_upernet",
    "fivebillionpixels": "seg_upernet",
}

# Which decoder to use per dataset (MLP fusion experiments)
MLP_DECODER = {
    "hlsburnscars":      "seg_mlp_fusion",
    "sen1floods11":      "seg_mlp_fusion",
    "mados":             "seg_mlp_fusion",
    "pastis":            "seg_mlp_fusion",
    "dynamicen":         "seg_mlp_fusion",
    "croptypemapping":   "seg_mlp_fusion",
    "spacenet7cd":       "seg_siam_mlp_fusion",
    "ai4smallfarms":     "seg_mlp_fusion",
    "fivebillionpixels": "seg_mlp_fusion",
}

# Display names and row order for the table
ENCODER_LABELS = {
    "dofa":               "DOFA",
    "dofav2":             "DOFAv2",
    "carl":               "CARL",
    "panopticon":         "Panopticon",
    "copernicus_fm_base": "Copernicus-FM",
    "smarties_vitb":      "SMARTIES",
    "dinov2_vitb14":      "DINOv2",
}

ENCODER_ORDER = list(ENCODER_LABELS.keys())

df_up  = pd.read_csv(UPERNET_LONG)
df_mlp = pd.read_csv(MLP_LONG)
print("UPerNet rows:", len(df_up))
print("MLP fusion rows:", len(df_mlp))


In [ ]:
def pivot_long(df, decoder_map):
    """
    Pivot the long CSV into an encoder × dataset matrix.
    For each (encoder, dataset) pair, keep only the run with the expected decoder,
    then take the best test_mIoU if multiple runs exist.
    """
    rows = []
    for ds, expected_dec in decoder_map.items():
        sub = df[(df["dataset"] == ds) & (df["decoder"] == expected_dec)].copy()
        sub["test_mIoU"] = pd.to_numeric(sub["test_mIoU"], errors="coerce")
        # best run per encoder (in case of reruns)
        best = sub.groupby("encoder")["test_mIoU"].max().reset_index()
        best["dataset"] = ds
        rows.append(best)

    long = pd.concat(rows, ignore_index=True)
    wide = long.pivot(index="encoder", columns="dataset", values="test_mIoU")
    wide = wide.reindex(columns=DATASETS)  # enforce column order

    # Add mean over available (non-NaN) values
    wide["mean"] = wide[DATASETS].mean(axis=1, skipna=True)

    # Reorder rows to ENCODER_ORDER, filling any missing encoders with NaN
    wide = wide.reindex(ENCODER_ORDER)
    wide.index = [ENCODER_LABELS.get(e, e) for e in wide.index]
    wide.columns = [DATASET_LABELS.get(c, c) if c != "mean" else "Mean" for c in wide.columns]
    return wide

pivot_up  = pivot_long(df_up,  UPERNET_DECODER)
pivot_mlp = pivot_long(df_mlp, MLP_DECODER)

print("=== UPerNet pivot ===")
display(pivot_up.round(2))
print("\n=== MLP Fusion pivot ===")
display(pivot_mlp.round(2))


In [ ]:
def to_latex(df, caption, label):
    """
    Generate a LaTeX table matching the GeoBench style:
    - \resizebox{\textwidth}{!}
    - | column borders, \hline rows
    - \textbf{} on the best (single max) value per column
    - "--" for missing values
    """
    ds_cols = [c for c in df.columns if c != "Mean"]
    all_cols = ds_cols + ["Mean"]

    # Find the single best encoder per column
    best_enc = {}
    for col in all_cols:
        s = df[col].dropna()
        if len(s) > 0:
            best_enc[col] = s.idxmax()

    def fmt(v, col, enc):
        if pd.isna(v):
            return "--"
        s = f"{v:.2f}"
        if best_enc.get(col) == enc:
            s = f"\\textbf{{{s}}}"
        return s

    col_spec = "|l|" + "r|" * len(all_cols)
    header_parts = ["\\textbf{Model}"] + [f"\\textbf{{{c}}}" for c in all_cols]
    header = " & ".join(header_parts) + " \\\\"

    lines = [
        "\\begin{table}[h]",
        "\\centering",
        "\\resizebox{\\textwidth}{!}{%",
        f"\\begin{{tabular}}{{{col_spec}}}",
        "\\hline",
        header,
        "\\hline",
    ]

    for enc, row in df.iterrows():
        cells = [enc] + [fmt(row[col], col, enc) for col in all_cols]
        lines.append(" & ".join(f"{c:<14}" for c in cells) + " \\\\")

    lines += [
        "\\hline",
        "\\end{tabular}}",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        "\\end{table}",
    ]
    return "\n".join(lines)

latex_up  = to_latex(pivot_up,  "Segmentation results with SegUPerNet (RGB)", "tab:seg_upernet_rgb")
latex_mlp = to_latex(pivot_mlp, "Segmentation results with SegMLPFusion (RGB)", "tab:seg_mlpfusion_rgb")

print(latex_up)
print()
print(latex_mlp)


In [ ]:
# Save to .tex files
with open("../rgb_upernet_table.tex", "w") as f:
    f.write(latex_up)
with open("../rgb_mlp_table.tex", "w") as f:
    f.write(latex_mlp)
print("Saved rgb_upernet_table.tex and rgb_mlp_table.tex")
